# Notebook 10 — Capstone 2: IMDB Sentiment (MLP vs. BERT)

**Module 3 · Text Classification — Capstone**
*Author: Axel Sirota — Data Trainers LLC*

## Story (STAR)

- **Situation.** IMDB movie reviews — 25K train, 25K test, binary positive/negative labels. This is *the* classical text classification benchmark, the MNIST of NLP. Your manager wants a sentiment classifier in production by Friday. You have the skills from Notebooks 8 and 9, but which approach do you ship?
- **Task.** Build a working classifier. You have **two paths:** (A) MLP with Word2Vec embeddings (Notebook 8 style), or (B) fine-tune DistilBERT (Notebook 9 style). Pick one, defend the choice, implement end-to-end, evaluate, and write up a recommendation.
- **Action.** Load IMDB from HuggingFace `datasets`. Pick your path. Implement from scratch: tokenize, train, validate, test. Measure accuracy, F1, inference latency, memory footprint. Write a short engineering memo justifying your choice with hard numbers.
- **Result.** A trained sentiment classifier with test accuracy 88–92% (MLP) or 93–95% (BERT), plus a written recommendation that balances accuracy vs. latency vs. cost for your production constraint.

## Learning objectives (cumulative Module 3 capstone)

By the end of this notebook you will be able to:

1. Autonomously apply the patterns from Notebook 8 or Notebook 9 to a new dataset.
2. Reason about trade-offs: accuracy vs. inference latency vs. memory vs. training cost.
3. Use HuggingFace `datasets` to load canonical benchmarks (one-line replacement for `keras.datasets.imdb`).
4. Tokenize, build a DataLoader, train, validate, test — without scaffolding.
5. Write a concise engineering memo with numbers, not vibes.
6. Defend technical decisions with data.

## Prerequisites

- Notebooks 5–9, especially NB8 (MLP) and NB9 (BERT).
- PyTorch, HuggingFace, gensim, scikit-learn.

## Runtime

- **MLP path:** ~45 min on Colab GPU (or ~2h on CPU).
- **BERT path:** ~30 min on Colab GPU (requires GPU; don't try on CPU for BERT).

---

## The Challenge

You will implement **ONE** of the two paths below. Choose based on your learning goal or the production constraint you want to simulate.

> 💡 **Hint:** If you want to maximize learning, implement **both** paths (as the solution does) and compare head-to-head. But for the exercise, pick one and go deep.

## Section 0 — Environment Setup & Data Loading

First, install dependencies and load the IMDB dataset. This section is the same for both paths.

In [ ]:
# Install packages (same for both paths).
!pip install -q "torch>=2.1" "transformers>=4.40" "datasets>=2.19" "evaluate>=0.4" \
    "accelerate>=0.28" "gensim>=4.3" "scikit-learn>=1.3" "pandas>=2.0" \
    "matplotlib>=3.7" "seaborn>=0.13"

In [ ]:
# Imports (shared).
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset
import evaluate

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

from gensim.models import Word2Vec

warnings.filterwarnings("ignore")

# Reproducibility.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Using device    : {device}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

print("\nEnvironment ready.")

In [ ]:
# Load IMDB dataset from HuggingFace.
# - 25K train, 25K test, perfectly balanced (50/50 pos/neg).
# - Each review is a string (avg ~200 tokens), label is 0 (negative) or 1 (positive).
imdb = load_dataset('imdb')
print(f"Dataset structure: {imdb}")
print(f"\nTrain size: {len(imdb['train']):,}")
print(f"Test size : {len(imdb['test']):,}")
print(f"\nSample review (positive):")
print(imdb['train'][0]['text'][:300], "...")
print(f"Label: {imdb['train'][0]['label']}  (0=negative, 1=positive)")

## YOUR CHOICE — Pick One Path

### Path A: MLP with Word2Vec (Notebook 8 style)

**Pros:**
- Fast inference (<5ms on CPU for batch of 32)
- Small model size (~2–5 MB)
- Works well with 5K+ labeled examples
- Easy to deploy to edge / mobile

**Cons:**
- Lower accuracy (~88–90% on IMDB)
- Static embeddings — can't handle polysemy or context
- Needs manual vocabulary building and tokenization

**When to pick:** You need <10ms latency, small memory footprint, or you're deploying to edge devices.

---

### Path B: Fine-tune DistilBERT (Notebook 9 style)

**Pros:**
- Higher accuracy (~93–95% on IMDB)
- Contextual embeddings — understands word order, negation, polysemy
- Pretrained — works with 1K+ labeled examples
- HuggingFace Trainer makes it ~40 lines of code

**Cons:**
- Slower inference (~20–50ms on GPU for batch of 32, ~200ms on CPU)
- Large model size (~270 MB)
- Requires GPU for reasonable training time

**When to pick:** You need the best accuracy, have GPU budget, and can afford 20–50ms latency.

---

### 🧪 **YOUR TASK:** Fill in the markdown cell below with your choice and justification.

> Example:
>
> *I will implement **Path A (MLP)** because my production constraint is <10ms latency on CPU for a mobile app. I expect ~88–90% accuracy, which is acceptable for my use case (user-facing sentiment in a review widget, where 90% is good enough).*

### My Choice

**I will implement:** [Path A or Path B]

**Justification:** [YOUR ANSWER — 2–3 sentences explaining why this path fits your learning goal or production constraint]

---

## Section 1 — Path A: MLP with Word2Vec

If you chose Path A, implement this section. If you chose Path B, **skip to Section 2.**

### Step 1: Split train → train/val (80/20)

In [ ]:
# 🧪 LAB: Split imdb['train'] into train_df and val_df (80/20, stratified by label).
# Hint: convert to pandas, use train_test_split, then convert back to lists.

# train_texts = ...  # list of strings
# train_labels = ... # list of 0/1
# val_texts = ...
# val_labels = ...

# YOUR CODE

### Step 2: Train Word2Vec on IMDB corpus

In [ ]:
# 🧪 LAB: Train Word2Vec on the full IMDB corpus (train + test).
# Tokenize with .split(), vector_size=100, window=5, min_count=5, workers=4, epochs=5.

# from gensim.models import Word2Vec
# corpus = ...  # list of lists of tokens
# w2v = Word2Vec(...)
# print(f"Vocab size: {len(w2v.wv)}")

# YOUR CODE

### Step 3: Build vocab, tokenize, pad

In [ ]:
# 🧪 LAB: Build a {word: idx} vocab from train_texts, tokenize all splits, pad to MAX_LEN=200.
# Hint: use a vocab with most common 20K words, add <PAD>=0 and <UNK>=1.

# MAX_LEN = 200
# vocab = ...  # {word: idx}
# def tokenize_and_pad(texts, vocab, max_len): ...
# train_ids = tokenize_and_pad(train_texts, vocab, MAX_LEN)
# val_ids = ...
# test_ids = ...

# YOUR CODE

### Step 4: Build embedding matrix from Word2Vec

In [ ]:
# 🧪 LAB: Build an embedding matrix from Word2Vec for the vocab.
# Shape: (vocab_size, 100). For words not in W2V, use random init.

# embedding_matrix = np.zeros((len(vocab), 100))
# for word, idx in vocab.items():
#     if word in w2v.wv:
#         embedding_matrix[idx] = w2v.wv[word]
#     else:
#         embedding_matrix[idx] = np.random.randn(100) * 0.01

# YOUR CODE

### Step 5: Define MLP model

In [ ]:
# 🧪 LAB: Define IMDBClassifier(nn.Module).
# Architecture: Embedding(from matrix, freeze=False) → masked mean pool → Linear(100, 128) → ReLU → Dropout(0.4) → Linear(128, 2).
# Remember: IMDB is 2-class, so output is Linear(hidden, 2) + CrossEntropyLoss.

# class IMDBClassifier(nn.Module):
#     def __init__(self, embedding_matrix, hidden_dim=128, dropout=0.4):
#         ...
#     def forward(self, x, mask):
#         # x: (batch, seq_len), mask: (batch, seq_len)
#         # Embed, mean-pool (with mask), pass through MLP.
#         ...

# model = IMDBClassifier(embedding_matrix).to(device)

# YOUR CODE

### Step 6: Training loop with early stopping

In [ ]:
# 🧪 LAB: Write training loop. Max 10 epochs, early stopping patience=3.
# Optimizer: Adam, LR=1e-3. Loss: CrossEntropyLoss. Batch size: 64.
# Track best val F1, save best model.

# YOUR CODE

### Step 7: Evaluate on test set

In [ ]:
# 🧪 LAB: Evaluate on imdb['test']. Report accuracy, macro-F1, confusion matrix.
# Also measure inference latency on CPU for a batch of 32 reviews.

# YOUR CODE

---

## Section 2 — Path B: Fine-tune DistilBERT

If you chose Path B, implement this section. If you chose Path A, **skip this.**

### Step 1: Tokenize with AutoTokenizer

In [ ]:
# 🧪 LAB: Tokenize IMDB with DistilBERT tokenizer, max_length=128 (reviews are long).
# Use .map() on the HF Dataset.

# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
# def tokenize_fn(examples):
#     return tokenizer(examples['text'], padding=False, truncation=True, max_length=128)
# train_dataset = imdb['train'].map(tokenize_fn, batched=True)
# test_dataset = imdb['test'].map(tokenize_fn, batched=True)

# YOUR CODE

### Step 2: Configure TrainingArguments

In [ ]:
# 🧪 LAB: Configure TrainingArguments for 2 epochs, LR=2e-5, batch=16 (reviews longer than headlines).
# eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True, metric='f1'.

# from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification, DataCollatorWithPadding
# training_args = TrainingArguments(...)

# YOUR CODE

### Step 3: Fine-tune with Trainer

In [ ]:
# 🧪 LAB: Instantiate model, Trainer, and call trainer.train().

# model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
# trainer = Trainer(...)
# trainer.train()

# YOUR CODE

### Step 4: Evaluate on test

In [ ]:
# 🧪 LAB: Evaluate on test set. Report accuracy, F1.
# Also measure inference latency on GPU and on CPU for a batch of 32.

# test_results = trainer.evaluate(test_dataset)
# print(f"Test accuracy: {test_results['eval_accuracy']:.4f}")
# print(f"Test F1: {test_results['eval_f1']:.4f}")

# Latency measurement:
# sample_batch = ...  # 32 reviews
# t0 = time.time(); model(...); elapsed = time.time() - t0
# print(f"Latency (GPU, batch 32): {elapsed*1000:.1f} ms")

# YOUR CODE

---

## Section 3 — Engineering Memo

### 🧪 **YOUR FINAL TASK:** Fill in the template below with your results.

## Recommendation Memo

**Chosen model:** [MLP or BERT]

**Test accuracy:** [YOUR NUMBER]%

**Test macro-F1:** [YOUR NUMBER]

**Inference latency (CPU, batch 32):** [YOUR NUMBER] ms

**Memory footprint (model size):** [YOUR NUMBER] MB

**Reason to ship this model:**

[YOUR ANSWER — 2–3 sentences explaining why this model is the right choice for your production constraint.]

**Reason NOT to ship this model (trade-offs):**

[YOUR ANSWER — 1–2 sentences on what you're giving up by not choosing the other path.]

**Fallback plan if accuracy drops in production:**

[YOUR ANSWER — 1 sentence on what you'd try next: more data, different model, ensemble, active learning, etc.]

In [ ]:
# 🧪 LAB: Plot confusion matrix for your chosen model.
# Also print classification_report.

# from sklearn.metrics import confusion_matrix, classification_report
# cm = confusion_matrix(all_true_labels, all_pred_labels)
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
# plt.title("Confusion Matrix — IMDB Sentiment")
# plt.show()
# print(classification_report(...))

# YOUR CODE

## Wrap-up

### What you've accomplished

- Applied Notebook 8 or Notebook 9 patterns to a new dataset autonomously.
- Reasoned about accuracy vs. latency vs. memory trade-offs.
- Wrote an engineering memo with hard numbers.
- Built a production-ready sentiment classifier (or at least production-*adjacent*!).

### Key takeaways

| Metric | MLP (Path A) | BERT (Path B) |
|--------|--------------|---------------|
| Accuracy | ~88–90% | ~93–95% |
| Latency (CPU, batch 32) | <5ms | ~200ms |
| Latency (GPU, batch 32) | <5ms | ~20–50ms |
| Model size | 2–5 MB | 270 MB |
| Training time | 10–20 min | 20–40 min |
| When to use | Edge, mobile, latency-critical | Cloud, GPU available, need best accuracy |

### Self-check quiz

1. With IMDB at 25K training examples, which model (MLP or BERT) wins on accuracy? Did your experiment match the expected outcome?
2. What would change if the dataset was 500 examples instead of 25K? Which model would you pick and why?
3. Your BERT model has 5× better F1 but 50× slower inference. The PM wants to ship in a mobile app. Which do you pick and why?

**Answers:**

1. BERT wins on accuracy (~93–95% vs ~88–90%). This matches expectation because BERT's contextual embeddings handle negation (*"not good"*) and polysemy better than static Word2Vec.
2. With 500 examples, MLP or LogReg would win. BERT would overfit. You'd need heavy regularization (dropout=0.5, weight_decay=0.1) or data augmentation to make BERT viable.
3. Pick MLP. Mobile apps are latency-sensitive and memory-constrained. 88–90% F1 is often "good enough" for user-facing features. If you *need* BERT-level accuracy, consider (a) distilling BERT into a 3-layer model, or (b) running inference server-side.

### What's next

Module 4 (Text Generation) starts with LSTMs for language modeling, then moves to GPT-style autoregressive generation. You'll see how the same transformer architecture (just swap encoder for decoder) powers text generation.

— *Capstone 2 complete.* 🚀